# ML Baseline — SVM, Logistic Regression, Random Forest

**Notebook:** `05_ml_baseline.ipynb`  
**Peneliti:** Khoeru Roziqin  
**Tanggal:** 03-05-26  

---

## Deskripsi

Notebook ini melatih tiga model **Machine Learning klasik** sebagai baseline pembanding:

| Model | Feature | Keterangan |
|---|---|---|
| **SVM** | TF-IDF | Support Vector Machine dengan kernel linear |
| **Logistic Regression** | TF-IDF | Multi-class (OvR) dengan regularisasi L2 |
| **Random Forest** | TF-IDF | Ensemble 200 decision trees |

### Tujuan
Mengukur seberapa jauh model berbasis Transformer (IndoBERT+CNN dan IndoBERT-only) melampaui pendekatan ML klasik pada task analisis sentimen MBG. Perbedaan performa antara ML baseline dan DL model merupakan bukti nilai tambah penggunaan pre-trained language model.

### Prinsip Fair Comparison
- **Split data identik** dengan notebook 03 dan 04 — menggunakan `test_set_indices.npy` yang sama
- **Input teks identik** — menggunakan `text_bert` (output preprocessing Step 1–4)
- **Metrik evaluasi identik** — F1-Macro sebagai metrik utama
- **Kondisi data identik** — S1 (class weighting) sesuai kondisi terbaik notebook 03

### Mengapa TF-IDF bukan IndoBERT embedding?
TF-IDF adalah feature representation standar untuk ML klasik — menggunakan IndoBERT embedding untuk ML baseline justru tidak fair karena "mencuri" keunggulan Transformer untuk model yang bukan Transformer. TF-IDF merepresentasikan kemampuan ML klasik secara mandiri.

---

## Pipeline

```
text_bert
    ↓
TF-IDF Vectorizer
  (max_features=50.000, ngram_range=(1,2), sublinear_tf=True)
    ↓
┌─────────────────────────────────────────┐
│ SVM          (kernel=linear, C=1.0)     │
│ Log. Regress (C=1.0, max_iter=1000)     │
│ Random Forest (n_estimators=200)        │
└─────────────────────────────────────────┘
    ↓
Stratified K-Fold 5 + Evaluasi Test Set
```

**Referensi:**  
- Cortes & Vapnik (1995). Support-vector networks. *Machine Learning*.  
- Cox (1958). The regression analysis of binary sequences. *JRSS-B*.  
- Breiman (2001). Random Forests. *Machine Learning*.  
- Salton & Buckley (1988). Term-weighting approaches. *Information Processing*.  

## 0. Setup

> Default: **Google Colab**. Untuk Kaggle: uncomment sel Kaggle, comment sel Colab.  
> Notebook ini **tidak membutuhkan GPU** — semua model ML berjalan di CPU.

In [ ]:
import sys, os

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')

if IS_KAGGLE:
    print('Environment: Kaggle Notebook')
elif IS_COLAB:
    print('Environment: Google Colab')
else:
    print('Environment: Local / Unknown')

In [ ]:
# ══════════════════════════════════════════════════════
#  SETUP — KAGGLE NOTEBOOK
# ══════════════════════════════════════════════════════

import os
print('✅ Setup Kaggle selesai. (CPU — tidak perlu GPU untuk ML baseline)')

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils import resample
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
import joblib
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
print(f'scikit-learn tersedia.')
import sklearn; print(f'sklearn version: {sklearn.__version__}')

## 1. Konfigurasi Global

## 2. Load & Split Data

Menggunakan **`test_set_indices.npy` yang sama** dengan notebook 03 dan 04.

In [ ]:
# ── PATH — KAGGLE ────────
INPUT_PATH      = '/kaggle/input/datasets/khoeruroziqin/final-mbg-labeled/final_mbg_labeled.csv'
NB03_OUTPUT_DIR = '/kaggle/working/model_results'
OUTPUT_DIR      = '/kaggle/working/baseline_results/ml'
MODEL_DIR       = '/kaggle/working/saved_models'
OUTPUTS_INPUT   = '/kaggle/input/datasets/khoeruroziqin/mbg-training-outputs'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR,  exist_ok=True)

# ── Restore test_set_indices.npy dari notebook 03 (Kaggle only) ───────────────
import shutil
_files = ['test_set_indices.npy', 'test_set_fixed.csv']
os.makedirs(NB03_OUTPUT_DIR, exist_ok=True)
print('Restoring files dari mbg-training-outputs...')
for fname in _files:
    src = f'{OUTPUTS_INPUT}/{fname}'
    dst = f'{NB03_OUTPUT_DIR}/{fname}'
    if os.path.exists(src):
        shutil.copy(src, dst); print(f'  ✔ {fname}')
    else:
        print(f'  ⚠️  Tidak ditemukan: {fname}')
print('Done.')

# ── Konfigurasi ───────────────────────────────────────────────────────────────
NUM_CLASSES = 3
LABEL2ID    = {'positive': 0, 'negative': 1, 'neutral': 2}
ID2LABEL    = {0: 'positive', 1: 'negative', 2: 'neutral'}
LABEL_NAMES = ['positive', 'negative', 'neutral']
N_FOLDS     = 5
DATA_COND   = 'S1'   # identik dengan kondisi terbaik notebook 03

# ── TF-IDF Konfigurasi ────────────────────────────────────────────────────────
TFIDF_CONFIG = {
    'max_features' : 50000,    # vocabulary size
    'ngram_range'  : (1, 2),   # unigram + bigram
    'sublinear_tf' : True,     # log(1+tf) — mengurangi dominasi kata frekuensi tinggi
    'min_df'       : 2,        # hapus term yang muncul < 2 dokumen
    'analyzer'     : 'word',
    'strip_accents' : 'unicode',
}

# ── Model Konfigurasi ─────────────────────────────────────────────────────────
# SVM: LinearSVC lebih cepat dari SVC(kernel='linear') untuk dataset besar
# class_weight='balanced' menggantikan class weighting manual untuk ML
MODELS = {
    'SVM': {
        'model'  : LinearSVC(C=1.0, max_iter=2000, random_state=SEED,
                             class_weight='balanced'),
        'desc'   : 'Linear SVM (C=1.0, class_weight=balanced)',
    },
    'LogisticRegression': {
        'model'  : LogisticRegression(C=1.0, max_iter=1000, solver='saga',
                                      multi_class='ovr', random_state=SEED,
                                      class_weight='balanced', n_jobs=-1),
        'desc'   : 'Logistic Regression (C=1.0, OvR, L2, class_weight=balanced)',
    },
    'RandomForest': {
        'model'  : RandomForestClassifier(n_estimators=200, max_depth=None,
                                          random_state=SEED, class_weight='balanced',
                                          n_jobs=-1),
        'desc'   : 'Random Forest (n=200, class_weight=balanced)',
    },
}

print('✅ Konfigurasi berhasil dimuat.')
print(f'   Kondisi data : {DATA_COND} (class_weight=balanced, identik S1 notebook 03)')
print(f'   TF-IDF       : max_features={TFIDF_CONFIG["max_features"]:,}, '
      f'ngram={TFIDF_CONFIG["ngram_range"]}, sublinear_tf={TFIDF_CONFIG["sublinear_tf"]}')
print(f'   Model        : {list(MODELS.keys())}')
print(f'   K-Fold       : {N_FOLDS}')
print(f'\n   Note: Notebook ini tidak membutuhkan GPU.')

In [ ]:
print('Memuat data berlabel...')
df = pd.read_csv(INPUT_PATH, low_memory=False)
df = df[df['label'].isin(['positive','negative','neutral'])].copy()
df['label_id'] = df['label'].map(LABEL2ID)
df = df.dropna(subset=['text_bert','label_id']).reset_index(drop=True)

print(f'Total data berlabel: {len(df):,}')
for lbl, cnt in df['label'].value_counts().items():
    print(f'  {lbl:12s}: {cnt:,} ({cnt/len(df)*100:.1f}%)')

# Load test set indices dari notebook 03 — WAJIB sama persis
test_idx_path = f'{NB03_OUTPUT_DIR}/test_set_indices.npy'
assert os.path.exists(test_idx_path), (
    f'❌ test_set_indices.npy tidak ditemukan di {test_idx_path}!\n'
    'Pastikan file output notebook 03 sudah tersedia.'
)

test_indices = np.load(test_idx_path)
train_mask   = np.ones(len(df), dtype=bool)
train_mask[test_indices] = False
df_trainval  = df[train_mask].reset_index(drop=True)
df_test      = df.iloc[test_indices].reset_index(drop=True)

print(f'\nTest set index dimuat dari notebook 03 (identik).')
print(f'Train+Val : {len(df_trainval):,} | Test (FIXED): {len(df_test):,}')
print(f'\nDistribusi test set:')
for lbl, cnt in df_test['label'].value_counts().items():
    print(f'  {lbl:12s}: {cnt:,} ({cnt/len(df_test)*100:.1f}%)')

## 3. Utilities

Fungsi pendukung training dan evaluasi ML pipeline.

In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        'accuracy'   : round(accuracy_score(y_true, y_pred), 4),
        'precision'  : round(precision_score(y_true, y_pred, average='macro', zero_division=0), 4),
        'recall'     : round(recall_score(y_true, y_pred, average='macro', zero_division=0), 4),
        'f1_macro'   : round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4),
        'f1_weighted': round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 4),
    }


def apply_undersampling(X, y, seed=SEED):
    min_n = min(np.bincount(y.astype(int)))
    X_res, y_res = [], []
    for cls in np.unique(y):
        idx = np.where(y == cls)[0]
        s   = resample(idx, n_samples=min_n, random_state=seed, replace=False)
        X_res.append(X[s]); y_res.append(y[s])
    X_res, y_res = np.concatenate(X_res), np.concatenate(y_res)
    shuf = np.random.RandomState(seed).permutation(len(X_res))
    return X_res[shuf], y_res[shuf]


def run_ml_kfold(model_name, clf, X_tv, y_tv, tfidf_config,
                 data_condition='S1', n_folds=5):
    """
    Stratified K-Fold untuk satu model ML dengan TF-IDF pipeline.
    TF-IDF difit hanya pada train fold untuk menghindari data leakage.

    Returns: dict avg metrik + fold_details list
    """
    skf          = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    fold_metrics = []
    fold_details = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_tv, y_tv), 1):
        X_tr, y_tr   = X_tv[tr_idx], y_tv[tr_idx]
        X_val, y_val = X_tv[val_idx], y_tv[val_idx]

        if data_condition == 'S2':
            X_tr, y_tr = apply_undersampling(X_tr, y_tr)

        # Fit TF-IDF hanya pada train fold — cegah data leakage
        vectorizer = TfidfVectorizer(**tfidf_config)
        X_tr_vec   = vectorizer.fit_transform(X_tr)
        X_val_vec  = vectorizer.transform(X_val)

        # Fit model
        t_fit = time.time()
        clf.fit(X_tr_vec, y_tr)
        fit_time = time.time() - t_fit

        # Predict
        y_pred = clf.predict(X_val_vec)
        m      = compute_metrics(y_val, y_pred)
        per_class = f1_score(y_val, y_pred, average=None, zero_division=0)

        fold_details.append({
            'model'          : model_name,
            'fold'           : fold,
            'fit_time_sec'   : round(fit_time, 2),
            **{f'val_{k}': v for k, v in m.items()},
            **{f'f1_{LABEL_NAMES[i]}': round(per_class[i], 4) for i in range(NUM_CLASSES)}
        })
        fold_metrics.append(m)
        print(f'  ├─ Fold {fold}/{n_folds} → F1={m["f1_macro"]:.4f} | '
              f'Acc={m["accuracy"]:.4f} | fit={fit_time:.1f}s')

    keys = fold_metrics[0].keys()
    avg  = {k: round(np.mean([m[k] for m in fold_metrics]), 4) for k in keys}
    std  = {f'{k}_std': round(np.std([m[k] for m in fold_metrics]), 4) for k in keys}
    return {**avg, **std}, fold_details


def plot_confusion_matrix_pair(y_true, y_pred, model_name, save_prefix):
    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, data, fmt, title in [
        (axes[0], cm,      'd',   f'Confusion Matrix (count)\n{model_name}'),
        (axes[1], cm_norm, '.2%', f'Confusion Matrix (normalized %)\n{model_name}')
    ]:
        sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                    xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(f'{save_prefix}_cm_pair.png', dpi=150, bbox_inches='tight')
    plt.close()


print('✅ Utilities berhasil didefinisikan.')
print('   TF-IDF difit hanya pada train fold — tidak ada data leakage.')

## 4. Training & Evaluasi Semua Model ML

Menjalankan K-Fold 5 dan evaluasi final test set untuk ketiga model secara berurutan.

> ⏱️ Estimasi waktu: **~10–20 menit** total (CPU). Random Forest paling lama.

In [ ]:
X_tv = df_trainval['text_bert'].values
y_tv = df_trainval['label_id'].values
X_te = df_test['text_bert'].values
y_te = df_test['label_id'].values

all_results      = []   # ringkasan per model untuk notebook 06
all_fold_details = []   # detail per fold semua model
t_start_global   = time.time()

for model_name, model_cfg in MODELS.items():
    print(f'\n{"═"*60}')
    print(f' MODEL: {model_name}')
    print(f' Deskripsi: {model_cfg["desc"]}')
    print(f'{"═"*60}')

    from sklearn.base import clone
    clf = clone(model_cfg['model'])

    # ── K-Fold Cross Validation ───────────────────────────────────────────────
    print(f'\n K-Fold {N_FOLDS} pada {len(df_trainval):,} train+val data...')
    t_kfold = time.time()
    kfold_avg, fold_details = run_ml_kfold(
        model_name, clf, X_tv, y_tv,
        tfidf_config   = TFIDF_CONFIG,
        data_condition = DATA_COND,
        n_folds        = N_FOLDS
    )
    kfold_time = time.time() - t_kfold
    all_fold_details.extend(fold_details)

    print(f'  └─ K-Fold selesai — {kfold_time/60:.1f} menit')
    print(f'     F1-Macro: {kfold_avg["f1_macro"]:.4f} ± {kfold_avg["f1_macro_std"]:.4f}')

    # ── Final Training pada seluruh train+val ──────────────────────────────────
    print(f'\n Final training pada seluruh {len(df_trainval):,} train+val...')

    # TF-IDF difit pada seluruh train+val
    vectorizer_final = TfidfVectorizer(**TFIDF_CONFIG)
    X_tv_vec = vectorizer_final.fit_transform(X_tv)
    X_te_vec = vectorizer_final.transform(X_te)

    clf_final = clone(model_cfg['model'])
    t_final   = time.time()
    clf_final.fit(X_tv_vec, y_tv)
    final_time = time.time() - t_final

    # ── Evaluasi final pada test set ──────────────────────────────────────────
    y_pred_te    = clf_final.predict(X_te_vec)
    final_metrics = compute_metrics(y_te, y_pred_te)
    per_class     = f1_score(y_te, y_pred_te, average=None, zero_division=0)

    # Confidence scores (probability)
    if hasattr(clf_final, 'predict_proba'):
        probs = clf_final.predict_proba(X_te_vec)
    else:
        # LinearSVC tidak punya predict_proba — gunakan decision_function
        dec   = clf_final.decision_function(X_te_vec)
        probs = np.exp(dec) / np.exp(dec).sum(axis=1, keepdims=True)

    print(f'\n{"─"*55}')
    print(f' EVALUASI FINAL — TEST SET ({model_name})')
    print(f'{"─"*55}')
    for k, v in final_metrics.items(): print(f'  {k:15s}: {v:.4f}')
    print(f'\n{classification_report(y_te, y_pred_te, target_names=LABEL_NAMES)}')
    print(f' Per-class F1: ' + ' | '.join(
        [f'{LABEL_NAMES[i]}={per_class[i]:.4f}' for i in range(NUM_CLASSES)]))

    # ── Confusion matrix pair ─────────────────────────────────────────────────
    fname_prefix = model_name.lower().replace(' ','_')
    plot_confusion_matrix_pair(
        y_te, y_pred_te,
        model_name  = f'{model_name} | F1-Macro={final_metrics["f1_macro"]:.4f}',
        save_prefix = f'{OUTPUT_DIR}/{fname_prefix}'
    )
    print(f'  ✔ Confusion matrix disimpan.')

    # ── Test predictions CSV ──────────────────────────────────────────────────
    df_pred = df_test[['full_text','text_bert','label']].copy() \
              if 'full_text' in df_test.columns \
              else df_test[['text_bert','label']].copy()
    df_pred['predicted']     = [ID2LABEL[p] for p in y_pred_te]
    df_pred['correct']       = df_pred['label'] == df_pred['predicted']
    df_pred['confidence']    = np.round(np.max(probs, axis=1), 4)
    df_pred['prob_positive'] = np.round(probs[:, 0], 4)
    df_pred['prob_negative'] = np.round(probs[:, 1], 4)
    df_pred['prob_neutral']  = np.round(probs[:, 2], 4)
    df_pred.to_csv(f'{OUTPUT_DIR}/{fname_prefix}_test_predictions.csv',
                   index=False, encoding='utf-8-sig')
    print(f'  ✔ {fname_prefix}_test_predictions.csv ({len(df_pred):,} baris)')

    # ── Simpan model & vectorizer ──────────────────────────────────────────────
    joblib.dump(clf_final,        f'{MODEL_DIR}/{fname_prefix}_model.pkl')
    joblib.dump(vectorizer_final, f'{MODEL_DIR}/{fname_prefix}_tfidf.pkl')
    print(f'  ✔ Model & TF-IDF disimpan (.pkl)')

    # ── Kumpulkan ringkasan ───────────────────────────────────────────────────
    all_results.append({
        'model'                  : model_name,
        'description'            : model_cfg['desc'],
        'feature'                : 'TF-IDF',
        'data_condition'         : DATA_COND,
        'kfold_f1_macro_mean'    : kfold_avg['f1_macro'],
        'kfold_f1_macro_std'     : kfold_avg['f1_macro_std'],
        'kfold_f1_weighted_mean' : kfold_avg['f1_weighted'],
        'kfold_accuracy_mean'    : kfold_avg['accuracy'],
        'test_accuracy'          : final_metrics['accuracy'],
        'test_precision'         : final_metrics['precision'],
        'test_recall'            : final_metrics['recall'],
        'test_f1_macro'          : final_metrics['f1_macro'],
        'test_f1_weighted'       : final_metrics['f1_weighted'],
        'f1_positive'            : round(per_class[0], 4),
        'f1_negative'            : round(per_class[1], 4),
        'f1_neutral'             : round(per_class[2], 4),
        'final_train_time_sec'   : round(final_time, 2),
        'kfold_total_time_min'   : round(kfold_time/60, 2),
    })

# ── Simpan semua hasil ────────────────────────────────────────────────────────
df_results     = pd.DataFrame(all_results)
df_fold_details = pd.DataFrame(all_fold_details)
df_results.to_csv(f'{OUTPUT_DIR}/ml_baseline_summary.csv', index=False)
df_fold_details.to_csv(f'{OUTPUT_DIR}/ml_baseline_fold_details.csv', index=False)

total_time = time.time() - t_start_global
print(f'\n{"═"*60}')
print(f' SEMUA MODEL SELESAI — {total_time/60:.1f} menit total')
print(f'{"═"*60}')
print(df_results[['model','test_f1_macro','test_accuracy',
                  'kfold_f1_macro_mean','kfold_f1_macro_std']].to_string(index=False))

## 5. Visualisasi Perbandingan Model ML

In [ ]:
df_res = pd.DataFrame(all_results)
df_fd  = pd.DataFrame(all_fold_details)
colors_model = ['#534AB7', '#E24B4A', '#1D9E75']
model_names  = df_res['model'].tolist()

# ── Plot 1: Perbandingan metrik test set ──────────────────────────────────────
metrics_compare = ['test_f1_macro','test_f1_weighted','test_accuracy',
                   'test_precision','test_recall']
labels_compare  = ['F1-Macro','F1-Weighted','Accuracy','Precision','Recall']
x = np.arange(len(labels_compare)); w = 0.25

fig, ax = plt.subplots(figsize=(14, 6))
for mi, (row, color) in enumerate(zip(df_res.itertuples(), colors_model)):
    vals = [getattr(row, m) for m in metrics_compare]
    ax.bar(x + mi*w - w, vals, w, label=row.model,
           color=color, edgecolor='none', alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels(labels_compare, fontsize=10)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Perbandingan Metrik Test Set — ML Baseline', fontsize=12)
ax.set_ylim(0.7, 1.0)
ax.legend(fontsize=10)
sns.despine(); plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/ml_comparison_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Plot 2: Stabilitas K-Fold (box plot per model) ────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
bp_data   = [df_fd[df_fd['model']==m]['val_f1_macro'].values for m in model_names]
bp_result = ax.boxplot(bp_data, patch_artist=True, widths=0.5)
for patch, color in zip(bp_result['boxes'], colors_model):
    patch.set_facecolor(color); patch.set_alpha(0.7)
for median in bp_result['medians']:
    median.set_color('white'); median.set_linewidth(2)
ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylabel('F1-Macro per Fold', fontsize=11)
ax.set_title('Stabilitas K-Fold — ML Baseline', fontsize=12)
ax.set_ylim(0.7, 1.0)
sns.despine(); plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/ml_kfold_stability.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Plot 3: Per-class F1 per model ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
x_cls = np.arange(NUM_CLASSES); w_cls = 0.25
for mi, (row, color) in enumerate(zip(df_res.itertuples(), colors_model)):
    per_cls = [row.f1_positive, row.f1_negative, row.f1_neutral]
    ax.bar(x_cls + mi*w_cls - w_cls, per_cls, w_cls,
           label=row.model, color=color, edgecolor='none', alpha=0.9)
ax.set_xticks(x_cls)
ax.set_xticklabels(LABEL_NAMES, fontsize=10)
ax.set_ylabel('F1-Score per Kelas', fontsize=11)
ax.set_title('Per-class F1 — ML Baseline', fontsize=12)
ax.set_ylim(0.6, 1.0)
ax.legend(fontsize=10)
sns.despine(); plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/ml_perclass_f1.png', dpi=150, bbox_inches='tight')
plt.show()

print('✔ Semua visualisasi ML baseline tersimpan.')

## 6. Ringkasan & Persiapan untuk Notebook 06

In [ ]:
# ── Tampilkan tabel ringkasan ─────────────────────────────────────────────────
print(f'{"═"*65}')
print(f' RINGKASAN ML BASELINE — TEST SET')
print(f'{"═"*65}')
print(f' {"Model":<22} {"F1-Macro":>10} {"F1-Wgt":>10} {"Accuracy":>10}')
print(f'{"─"*65}')
for row in df_res.itertuples():
    print(f' {row.model:<22} {row.test_f1_macro:>10.4f} '
          f'{row.test_f1_weighted:>10.4f} {row.test_accuracy:>10.4f}')
print(f'{"─"*65}')
best_ml = df_res.loc[df_res['test_f1_macro'].idxmax()]
print(f' Best ML: {best_ml["model"]} (F1-Macro={best_ml["test_f1_macro"]:.4f})')
print(f'{"═"*65}')

# ── Per-class F1 ──────────────────────────────────────────────────────────────
print(f'\n Per-class F1 (test set):')
print(f' {"Model":<22} {"Positive":>10} {"Negative":>10} {"Neutral":>10}')
print(f'{"─"*55}')
for row in df_res.itertuples():
    print(f' {row.model:<22} {row.f1_positive:>10.4f} '
          f'{row.f1_negative:>10.4f} {row.f1_neutral:>10.4f}')

# ── Perbandingan dengan DL models ─────────────────────────────────────────────
proposed_f1   = 0.8453   # dari notebook 03
bert_only_f1  = 0.8497   # dari notebook 04

print(f'\n{"═"*65}')
print(f' PERBANDINGAN AWAL — ML vs DL')
print(f'{"═"*65}')
print(f' IndoBERT+CNN (proposed) : {proposed_f1:.4f}')
print(f' IndoBERT-only (baseline): {bert_only_f1:.4f}')
for row in df_res.itertuples():
    delta = proposed_f1 - row.test_f1_macro
    print(f' {row.model:<25}: {row.test_f1_macro:.4f} '
          f'(proposed lebih baik {delta:+.4f})')
print(f'{"═"*65}')

# ── Output yang dihasilkan ────────────────────────────────────────────────────
print(f'\n Output yang dihasilkan:')
outputs = [
    (OUTPUT_DIR, 'ml_baseline_summary.csv',              'Ringkasan metrik semua model untuk nb06'),
    (OUTPUT_DIR, 'ml_baseline_fold_details.csv',         'Detail metrik per fold semua model'),
    (OUTPUT_DIR, 'ml_comparison_metrics.png',            'Grouped bar chart perbandingan metrik'),
    (OUTPUT_DIR, 'ml_kfold_stability.png',               'Box plot stabilitas K-Fold'),
    (OUTPUT_DIR, 'ml_perclass_f1.png',                   'Per-class F1 per model'),
    (OUTPUT_DIR, 'svm_cm_pair.png',                      'Confusion matrix SVM'),
    (OUTPUT_DIR, 'logisticregression_cm_pair.png',       'Confusion matrix LR'),
    (OUTPUT_DIR, 'randomforest_cm_pair.png',             'Confusion matrix RF'),
    (OUTPUT_DIR, 'svm_test_predictions.csv',             'Prediksi SVM per tweet'),
    (OUTPUT_DIR, 'logisticregression_test_predictions.csv', 'Prediksi LR per tweet'),
    (OUTPUT_DIR, 'randomforest_test_predictions.csv',    'Prediksi RF per tweet'),
    (MODEL_DIR,  'svm_model.pkl',                        'Model SVM final'),
    (MODEL_DIR,  'svm_tfidf.pkl',                        'TF-IDF SVM final'),
    (MODEL_DIR,  'logisticregression_model.pkl',         'Model LR final'),
    (MODEL_DIR,  'logisticregression_tfidf.pkl',         'TF-IDF LR final'),
    (MODEL_DIR,  'randomforest_model.pkl',               'Model RF final'),
    (MODEL_DIR,  'randomforest_tfidf.pkl',               'TF-IDF RF final'),
]
for d, fname, desc in outputs:
    exists = '✔' if os.path.exists(f'{d}/{fname}') else '○'
    print(f'  {exists} {fname:<50} {desc}')

print(f'\n Lanjut ke Notebook 06 — Model Comparison')

---

## Catatan Metodologis untuk Laporan

### Feature Extraction: TF-IDF
TF-IDF (Term Frequency-Inverse Document Frequency) dengan skema `sublinear_tf=True` menggunakan $\log(1 + tf)$ untuk mengurangi dominasi kata frekuensi tinggi. N-gram (1,2) menangkap pola unigram dan bigram — relevan untuk sentimen Twitter yang sering menggunakan frasa pendek bermakna.

### Pencegahan Data Leakage
TF-IDF difit **hanya pada data train fold** di setiap iterasi K-Fold, kemudian ditransform ke val fold. Ini mencegah informasi vocabulary dari val fold bocor ke proses training — konsisten dengan praktik standar dalam evaluasi NLP.

### Penanganan Imbalanced Data
Parameter `class_weight='balanced'` pada ketiga model ML secara otomatis menghitung bobot kelas berbanding terbalik dengan frekuensinya — fungsional setara dengan class weighting pada CrossEntropyLoss di notebook 03 dan 04.

### Pemilihan Model ML
- **LinearSVC**: Efisien untuk high-dimensional sparse feature (TF-IDF), terbukti kompetitif pada text classification (Joachims, 1998)
- **Logistic Regression**: Baseline probabilistik yang kuat, interpretable, dengan regularisasi L2
- **Random Forest**: Ensemble non-linear yang robust terhadap noise dan outlier

### Referensi
- Cortes & Vapnik (1995). Support-vector networks. *Machine Learning*, 20, 273–297.
- Breiman, L. (2001). Random Forests. *Machine Learning*, 45, 5–32.
- Joachims, T. (1998). Text categorization with SVM. *ECML 1998*.
- Salton & Buckley (1988). Term-weighting approaches in automatic text retrieval. *Information Processing & Management*.